# Collect state recall and advisory data
Start with one public recall or advisory page. This notebook shows what the server sends, inspects the export controls a browser sees, and saves the extracted evidence as one CSV per state. Run the cells from top to bottom with **Shift + Enter**. To build it yourself, recreate sections 1–3 in a blank notebook first.

**Research scope.** A product row, a notice and a recall event are different units. The Massachusetts example describes marijuana products and public health advisories. Keep original labels, then verify the product origin and action type against the notice before counting hemp-derived recalls. [Massachusetts source](https://masscannabiscontrol.com/news/public-health-and-safety-advisories/)

## 1. Set up your environment
On Windows, open **Miniforge Prompt**, run these commands, and open this notebook in the JupyterLab window. Select the Python kernel from this environment.

```sh
python -m pip install playwright
python -m playwright install chromium
```

Pandas reads the spreadsheets, lxml reads HTML, pdfplumber reads PDF pages, and Playwright opens Chromium to find and click export buttons. The code uses a worker thread for the browser so it runs inside a Windows Jupyter kernel.

In [1]:
from pathlib import Path
from urllib.parse import urljoin, urlsplit, unquote
from urllib.request import Request, urlopen
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
from io import BytesIO
from collections import Counter, defaultdict
import hashlib
import re

import pandas as pd
import pdfplumber
from lxml import html
from playwright.sync_api import sync_playwright

## 2. Enter your source
Use the page containing the recall list or an individual file URL. Each state can have several URLs. For your first run, leave just Massachusetts. A second example is commented out so it cannot run accidentally.

In [2]:
SOURCES = {
    "Massachusetts": [
        "https://masscannabiscontrol.com/news/public-health-and-safety-advisories/"
    ],
    # "Another State": ["PASTE_THE_OFFICIAL_RECALL_URL_HERE"],
}
OUTPUT = Path("recall_outputs")
STATE = next(iter(SOURCES))
URL = SOURCES[STATE][0]

## 3. Look at what Python receives
`fetch` follows redirects and returns the response body, final address, and headers. Try the next two cells with the Massachusetts example. A successful HTTP response alone tells you nothing about which data are present.

In [3]:
def fetch(url):
    if urlsplit(url).scheme not in {"http", "https"}:
        raise ValueError("Enter a full http or https URL.")
    request = Request(url, headers={"User-Agent": "Mozilla/5.0 (state recall research)"})
    with urlopen(request, timeout=60) as response:
        return response.read(), response.geturl(), response.headers

def document(body):
    # Preserve explicit UTF-8 if present. Let lxml detect other encodings.
    try:
        body.decode("utf-8-sig")
        parser = html.HTMLParser(encoding="utf-8")
    except UnicodeDecodeError:
        parser = html.HTMLParser()
    return html.fromstring(body, parser=parser)

def suffix_hint(text):
    match = re.search(r"\b(csv|xlsx|xls|excel|pdf)\b", unquote(text), re.I)
    if match:
        word = match.group(1).lower()
        return "xlsx" if word == "excel" else word

def file_kind(url, headers, body):
    if body.lstrip().startswith(b"%PDF-"):
        return "pdf"
    if body.startswith(b"PK\x03\x04"):
        return "xlsx"
    if body.startswith(b"\xd0\xcf\x11\xe0"):
        return "xls"
    if "html" in headers.get("Content-Type", "").lower() or body.lstrip()[:100].lower().startswith((b"<!doctype html", b"<html")):
        return None
    return suffix_hint(headers.get("Content-Disposition", "") + " " +
                       headers.get("Content-Type", "") + " " + urlsplit(url).path)

In [4]:
body, address, headers = fetch(URL)
kind = file_kind(address, headers, body)
print("Final address:", address)
if kind:
    print("The URL returned a file:", kind)
else:
    tree = document(body)
    print("Page heading:", " ".join(tree.xpath("//h1//text()")).strip())
    print("HTML tables:", len(tree.xpath("//table")))
    for table in tree.xpath("//table"):
        print("First row:", table.xpath("(.//tr)[1]/*/descendant-or-self::*/text()")[:12])

Final address: https://masscannabiscontrol.com/news/public-health-and-safety-advisories/
Page heading: Public Health and Safety Advisories Attend a Public Meeting Subscribe to Our Notice List Attend a Public Meeting Subscribe to Our Notice List
HTML tables: 1
First row: ['Package Label', 'Packaged Date', 'Tested On', 'Product Name', 'Product Category', 'Harvest Name', 'Sold Between Dates', 'Advisory Date']


### Find direct file links
An anchor such as `<a href="recalls.csv">` gives you an address to download. Some agencies use extensionless addresses with a label such as “Download Excel,” so we also inspect the link text and download filename. Look at the printed titles for unrelated files. Add distinctive title or URL fragments to `IGNORE` as needed.

In [5]:
IGNORE = ["understanding labels"]  # A general Massachusetts flyer, outside the dataset.

def direct_files(tree, base):
    found = {}
    for link in tree.xpath("//a[@href]"):
        href = urljoin(base, link.get("href"))
        label = " ".join(link.text_content().split())
        hint = suffix_hint(urlsplit(href).path) or suffix_hint(
            " ".join([label, link.get("download", ""), link.get("title", ""), href.partition("?")[2]])
        )
        if urlsplit(href).scheme not in {"https", "http"} or href.split("#")[0] == base.split("#")[0]:
            continue
        if hint and not any(term.lower() in (label + " " + href).lower() for term in IGNORE):
            found[href] = (label or hint.upper(), hint)
    return found

if not kind:
    links = direct_files(tree, address)
    print(f"Found {len(links)} direct file links")
    for href, (label, fmt) in links.items():
        print(fmt, label, href)

Found 0 direct file links


## 4. Find buttons in a real browser
A button can generate a CSV in the browser without putting a CSV link in the original HTML. This is why the previous cell can correctly report zero links. The browser helper reads visible labels, titles, accessibility labels and common export-button classes. It checks embedded frames too.

On Massachusetts, the toolbar has CSS class `dt-buttons`. The CSV button has class `buttons-csv`. If you are writing your own notebook, the short Massachusetts-only example below is a good first browser experiment.

Run the **preview** cell after these definitions. It lists candidate controls without clicking them. In the collection step, the script starts listening for a download **before** clicking. Within a shared export toolbar, it tries CSV first, Excel second and PDF third, stopping once a readable export succeeds. Separate toolbars are processed independently. [Playwright downloads](https://playwright.dev/python/docs/downloads)

In [6]:
CONTROL_SCAN = """() => {
    const nodes = document.querySelectorAll(
        'button, a, [role="button"], input[type=button], input[type=submit]'
    );
    const bars = [...document.querySelectorAll('.dt-buttons, [role=toolbar]')];
    const found = [];
    nodes.forEach((el, index) => {
        if (!el.getClientRects().length) return;
        const label = [el.innerText, el.value, el.title,
                       el.getAttribute('aria-label'), el.getAttribute('class')].filter(Boolean).join(' ');
        const match = label.match(/\\b(csv|xlsx|xls|excel|pdf)(?:html5)?\\b/i);
        if (!match) return;
        const href = el.getAttribute('href') || '';
        if (/^https?:/i.test(el.href || '') && href && !href.startsWith('#')
            && (el.href.split('#')[0] !== location.href.split('#')[0])) return;
        el.dataset.recallExportId = String(index);
        const bar = el.closest('.dt-buttons, [role=toolbar]');
        found.push({id: index, group: bar ? 'bar-' + bars.indexOf(bar) : 'control-' + index,
                    label: label.trim(), kind: match[1].toLowerCase()});
    });
    return found;
}"""

def in_browser(job, url):
    # Playwright's synchronous API needs a thread without Jupyter's event loop.
    def run():
        with sync_playwright() as play:
            browser = play.chromium.launch()
            page = browser.new_page(accept_downloads=True)
            try:
                page.goto(url, wait_until="load", timeout=60000)
                return job(page)
            finally:
                browser.close()
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(run).result()

def controls(frame):
    # Allow late JavaScript initialization without requiring an idle network.
    try:
        frame.wait_for_function("() => (" + CONTROL_SCAN + ")().length > 0", timeout=2500)
    except Exception:
        pass
    return frame.evaluate(CONTROL_SCAN)

def preview(page):
    for frame in page.frames:
        buttons = controls(frame)
        print(frame.url, "—", len(buttons), "export control(s)")
        for item in buttons:
            print(" ", item["group"], item["kind"], item["label"])

**Your first button experiment.** After running the definitions above, try this in a new cell for Massachusetts. The selector `.dt-buttons button` means “every button inside the table export toolbar.” You should see Copy, CSV, PDF, Print and Excel, possibly in a different order.

```python
def show_massachusetts_buttons(page):
    toolbar_buttons = page.locator(".dt-buttons button")
    toolbar_buttons.first.wait_for(state="attached", timeout=15000)
    print(toolbar_buttons.all_text_contents())

in_browser(show_massachusetts_buttons, URL)
```

The next runnable cell generalizes this check to other button styles and embedded frames.

In [7]:
if not kind:
    try:
        in_browser(preview, address)
    except Exception as error:
        print("Browser preview unavailable:", error)

https://masscannabiscontrol.com/news/public-health-and-safety-advisories/ — 3 export control(s)
  bar-0 csv CSV dt-button buttons-csv buttons-html5
  bar-0 pdf PDF dt-button buttons-pdf buttons-html5
  bar-0 excel Excel dt-button buttons-excel buttons-html5
https://www.youtube.com/embed/EQX-zxb_CBI?feature=oembed&autoplay=1&loop=1&controls=1&mute=1&wmode=opaque&rel=0&enablejsapi=1&origin=https%3A%2F%2Fmasscannabiscontrol.com — 0 export control(s)
https://www.google.com/recaptcha/enterprise/anchor?ar=1&k=6LdSXx4uAAAAALSr9T5Rg_RAsWA1s28fjg7VROcF&co=aHR0cHM6Ly9tYXNzY2FubmFiaXNjb250cm9sLmNvbTo0NDM.&hl=en&v=4i3aN5wAweL7i3ADI7BSX4B-&size=invisible&anchor-ms=20000&execute-ms=30000&ovv=CADq00gF&cb=x58l81d92lem — 0 export control(s)
about:blank — 0 export control(s)
about:blank — 0 export control(s)


## 5. Turn files into rows
The original columns stay intact. Each observation also receives a source URL, extraction method, and file, sheet, table or page reference. Empty strings stay empty and CSV fields are read as strings so batch numbers with leading zeros survive.

A spreadsheet yields rows for each worksheet. A PDF yields a text record per page and, when grid lines are detected, extra table rows. PDF table cells are numbered because continuation pages often omit the headers. Use the page text and raw PDF to interpret them. An image-only page gets a review flag for OCR. Filter by `record_type` when analyzing these outputs.

In [8]:
def table_records(frame, method, section):
    frame = frame.fillna("")
    return [dict(record_type=method, section=section, **{
        "data_" + str(key): value for key, value in row.items()
    }) for row in frame.to_dict("records") if any(str(v).strip() for v in row.values())]

def pdf_records(data):
    records = []
    with pdfplumber.open(BytesIO(data)) as pdf:
        for number, page in enumerate(pdf.pages, start=1):
            text = (page.extract_text() or "").strip()
            records.append(dict(record_type="pdf_page", section=f"page {number}",
                                data_text=text, extraction_note="" if text else "needs OCR or visual review"))
            for idx, table in enumerate(page.extract_tables(), start=1):
                frame = pd.DataFrame(table)
                frame.columns = [f"cell_{j + 1}" for j in range(len(frame.columns))]
                records.extend(table_records(frame, "pdf_table_row", f"page {number}, table {idx}"))
    return records

def file_records(data, fmt):
    if fmt == "pdf":
        return pdf_records(data)
    if fmt in {"xls", "xlsx", "excel"}:
        sheets = pd.read_excel(BytesIO(data), sheet_name=None, dtype=str, keep_default_na=False)
        return [r for sheet, frame in sheets.items() for r in table_records(frame, "excel_row", sheet)]
    for encoding in ["utf-8-sig", "cp1252"]:
        try:
            text = data.decode(encoding)
            break
        except UnicodeDecodeError:
            continue
    else:
        raise ValueError("Unrecognized CSV encoding")
    import csv
    from io import StringIO
    try:
        delimiter = csv.Sniffer().sniff(text[:8192], delimiters=",;\t|").delimiter
    except csv.Error:
        delimiter = ","
    frame = pd.read_csv(StringIO(text), sep=delimiter, dtype=str, keep_default_na=False)
    return table_records(frame, "csv_row", "CSV")

def keep_file(data, fmt, source, title, folder):
    digest = hashlib.sha256(data).hexdigest()[:16]
    path = folder / f"{digest}.{fmt}"
    path.write_bytes(data)
    rows = file_records(data, fmt)
    if not rows:
        raise ValueError(f"No rows in {path.name}")
    for row in rows:
        row.update(source_url=source, source_label=title, raw_file=path.name)
    return rows

## 6. Read HTML when an export is unavailable
Look for tables first. Preserve their column headers and the source's identifiers. If the page has no usable table, capture the text of each article or notice block and preserve a link for review. Some states use single-page lists without article tags. Set `NOTICE_XPATH` to the repeated list-item element for that site's layout, or enter the individual notice URLs in `SOURCES`. Use `TABLE_XPATH` to restrict a page with unrelated tables.

In [9]:
TABLE_XPATH = "//table"
NOTICE_XPATH = "//main//article | //article[not(ancestor::article)]"

def cells_of(row):
    return [" ".join(cell.text_content().split()) for cell in row.xpath("./th | ./td")]

def html_records(tree, url):
    rows = []
    for number, table in enumerate(tree.xpath(TABLE_XPATH), start=1):
        lines = [cells_of(row) for row in table.xpath("./tr | ./thead/tr | ./tbody/tr")]
        if len(lines) < 2:
            continue
        has_header = bool(table.xpath("(.//tr)[1]/th"))
        has_header = has_header or bool(table.xpath("./thead/tr"))
        values = lines[1:] if has_header else lines
        width = max(map(len, lines))
        names = list(lines[0]) if has_header else []
        names += [""] * (width - len(names))
        # Keep duplicate or blank column labels distinct.
        names = [name if name and names.count(name) == 1 else f"column_{i+1}_{name}"
                 for i, name in enumerate(names)]
        frame = pd.DataFrame(values, columns=names)
        section = table.get("id") or f"table {number}"
        rows.extend(table_records(frame, "html_table_row", section))
    if rows:
        return rows
    blocks = tree.xpath(NOTICE_XPATH) or tree.xpath("(//main | //*[@role='main'])[1]") or tree.xpath("//body")
    for number, block in enumerate(blocks, start=1):
        parts = block.xpath(".//text()[normalize-space() and not(ancestor::script or ancestor::style or ancestor::nav or ancestor::footer)]")
        text = " ".join(" ".join(parts).split())
        hrefs = block.xpath(".//h2//a/@href | .//h3//a/@href | self::a/@href")
        if text:
            rows.append(dict(record_type="html_text", section=f"block {number}", data_text=text,
                             detail_url=urljoin(url, hrefs[0]) if hrefs else "",
                             extraction_note="Review notice scope and linked details"))
    return rows

## 7. Connect discovery, download and fallback
A failed download is logged and the collector tries the next option. Each source table is checked separately so a successful export from one table can coexist with an HTML-only table on the same page.

Direct files are saved individually. Matching spreadsheet rows take priority over a duplicate HTML table. PDFs linked alongside a table are preserved separately. If there are multiple unrelated datasets on one page, check the preview and the output by section. Use more precise source URLs for missing sections. This notebook handles the supplied pages and their file links. Pagination, notice archives, export menus requiring extra clicks, and scanned PDFs may need a state-specific step.

In [10]:
def linked_rows(tree, original_tree, source, folder, direct_seen, diagnostics):
    links = direct_files(tree, source)
    if original_tree is not None:
        links.update(direct_files(original_tree, source))
    direct_rows = []
    for href, (label, hint) in links.items():
        if href in direct_seen:
            continue
        direct_seen.add(href)
        try:
            data, final, headers = fetch(href)
            fmt = file_kind(final, headers, data)
            if not fmt and ("html" in headers.get("Content-Type", "").lower() or data.lstrip().startswith(b"<")):
                raise ValueError("link opened a webpage; use its address as a source")
            direct_rows.extend(keep_file(data, fmt or hint, final, label, folder))
        except Exception as error:
            diagnostics.append(f"File {href}: {error}")

    return direct_rows

In [11]:
def exported_rows(page, frame, items_found, folder, diagnostics):
    button_rows = []
    groups = {}
    for item in items_found:
        groups.setdefault(item["group"], []).append(item)
    priority = {"csv": 0, "excel": 1, "xlsx": 1, "xls": 1, "pdf": 2}
    for group, items in groups.items():
        for item in sorted(items, key=lambda x: priority[x["kind"]]):
            try:
                selector = f'[data-recall-export-id="{item["id"]}"]'
                with page.expect_download(timeout=12000) as pending:
                    frame.locator(selector).click(timeout=6000)
                download = pending.value
                data = download.path().read_bytes()
                fmt = file_kind(download.suggested_filename, {}, data) or item["kind"]
                button_rows.extend(keep_file(data, fmt, frame.url, group + " " + item["label"], folder))
                break
            except Exception as error:
                diagnostics.append(f"Button {item['label']}: {error}")
    return button_rows

### Compare the fallback with the export
The same rows may appear both in a CSV and in the HTML. Group the HTML by table, select the more complete copy from the original and rendered page, and skip tables whose rows already appear in an export. When the contents differ, retain the table for review. Raw PDFs and page-text records have their own record types.

In [12]:
def signature(row):
    return tuple(sorted((key, str(value).strip()) for key, value in row.items() if key.startswith("data_")))

def combine_sources(downloads, originals, rendered, source, diagnostics):
    tables = {}
    for version in (originals, rendered):
        by_section = defaultdict(list)
        for row in version:
            if row["record_type"] == "html_table_row":
                by_section[row["section"]].append(row)
        for name, group in by_section.items():
            if len(group) > len(tables.get(name, [])):
                tables[name] = group

    exported = Counter(signature(row) for row in downloads if row["record_type"] in {"csv_row", "excel_row"})
    additional = []
    for name, group in tables.items():
        if Counter(signature(row) for row in group) - exported:
            additional.extend(group)
            if exported:
                diagnostics.append(f"Compare HTML {name} with the spreadsheet exports; their rows differ.")
    if not tables and not downloads:
        additional = rendered or originals
    for row in additional:
        row["source_url"] = source
    return downloads + additional

def capture_frame(page, frame, original, folder, seen, diagnostics):
    buttons = controls(frame)
    tree = document(frame.content().encode())
    source = frame.url
    downloads = linked_rows(tree, original, source, folder, seen, diagnostics)
    downloads.extend(exported_rows(page, frame, buttons, folder, diagnostics))
    original_rows = html_records(original, source) if original is not None else []
    return combine_sources(downloads, original_rows, html_records(tree, source), source, diagnostics)

In [13]:
def collect_page(url, folder, diagnostics):
    data, final, headers = fetch(url)
    fmt = file_kind(final, headers, data)
    if fmt:
        return keep_file(data, fmt, final, "direct URL", folder)
    original = document(data)
    def visit(page):
        rows, seen = [], set()
        for frame in page.frames:
            main = frame == page.main_frame
            if not main and (not frame.url.startswith("http") or not frame.locator("table, .dt-buttons, [role=toolbar]").count()):
                continue
            try:
                rows.extend(capture_frame(page, frame, original if main else None, folder, seen, diagnostics))
            except Exception as error:
                diagnostics.append(f"Frame {frame.url}: {error}")
        if not rows:
            raise ValueError("Browser yielded no records")
        return rows
    try:
        return in_browser(visit, final)
    except Exception as error:
        diagnostics.append(f"Using original HTML because browser collection failed: {error}")
        downloads = linked_rows(original, None, final, folder, set(), diagnostics)
        return combine_sources(downloads, html_records(original, final), [], final, diagnostics)

## 8. Save and inspect each state separately
Each run creates its own timestamped directory under **recall_outputs** so a failed or partial run cannot overwrite an earlier one. Each state has one data CSV, a **raw** folder for downloads, and a **review.txt** log when a source needs attention. The prefix **data_** identifies fields copied from the source. **hemp_status** and **action_type** start as **unreviewed**. A file hash identifies a downloaded document, including when two links offer the same bytes.

In [14]:
def run_state(state, urls, run_dir):
    slug = re.sub(r"[^a-z0-9]+", "_", state.lower()).strip("_")
    folder = run_dir / slug
    raw = folder / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    diagnostics, records = [], []
    for url in dict.fromkeys(urls):
        try:
            rows = collect_page(url, raw, diagnostics)
            for row in rows:
                row["page_url"] = url
            records.extend(rows)
            print(state, "—", len(rows), "evidence rows from", url)
        except Exception as error:
            diagnostics.append(f"Source {url}: {error}")
    if not records:
        diagnostics.append("No records collected. Inspect the source and its layout.")
    if diagnostics:
        (folder / "review.txt").write_text("\n".join(diagnostics), encoding="utf-8")
        print("Review:", folder / "review.txt")
    df = pd.DataFrame(records)
    if not df.empty:
        df.insert(0, "state", state)
        df["hemp_status"] = "unreviewed"
        df["action_type"] = "unreviewed"
        path = folder / f"{slug}_records.csv"
        df.to_csv(path, index=False, encoding="utf-8-sig")
        print("Saved:", path.resolve())
    return df

run_dir = OUTPUT / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
results = {state: run_state(state, urls, run_dir) for state, urls in SOURCES.items()}

Massachusetts — 7571 evidence rows from https://masscannabiscontrol.com/news/public-health-and-safety-advisories/
Saved: C:\Users\pegah\Code\Cannabis-Recalls-Analysis\recall_outputs\20261123T032419_362704Z\massachusetts\massachusetts_records.csv


In [15]:
sample = results[STATE]
if not sample.empty:
    display(sample.groupby(["record_type", "section"], dropna=False).size().rename("rows").head(30))
    display(sample.head(5))

record_type  section
csv_row      CSV        7571
Name: rows, dtype: int64

,state,record_type,section,data_Package Label,data_Packaged Date,data_Tested On,data_Product Name,data_Product Category,data_Harvest Name,data_Sold Between Dates,data_Advisory Date,source_url,source_label,raw_file,page_url,hemp_status,action_type
0,Massachusetts,csv_row,CSV,1A40A01000006A7000014404,2024-10-02,2024-09-24,M00002432108: Alpine Strawberry - 0.5g PRJs (5pk),Raw Pre-Rolls,240528-KSM-F5C12,2024-10-03 to 2024-10-03,08/06/2025,https://masscannabiscontrol.com/news/public-he...,bar-0 CSV dt-button buttons-csv buttons-html5,8a53b612d4922796.csv,https://masscannabiscontrol.com/news/public-he...,unreviewed,unreviewed
1,Massachusetts,csv_row,CSV,1A40A01000007D1000033992,2025-04-03,2025-02-03,M00003057781: Preroll Skywalker OG 1G Sativa,Raw Pre-Rolls,010625SWOG-HB7,2025-04-05 to 2025-04-10,08/06/2025,https://masscannabiscontrol.com/news/public-he...,bar-0 CSV dt-button buttons-csv buttons-html5,8a53b612d4922796.csv,https://masscannabiscontrol.com/news/public-he...,unreviewed,unreviewed
2,Massachusetts,csv_row,CSV,1A40A01000007D1000034004,2025-04-03,2025-02-03,M00003057781: Preroll Skywalker OG 1G Sativa,Raw Pre-Rolls,010625SWOG-HB7,2025-04-05 to 2025-04-17,08/06/2025,https://masscannabiscontrol.com/news/public-he...,bar-0 CSV dt-button buttons-csv buttons-html5,8a53b612d4922796.csv,https://masscannabiscontrol.com/news/public-he...,unreviewed,unreviewed
3,Massachusetts,csv_row,CSV,1A40A01000007D1000034020,2025-04-03,2025-02-03,M00003057781: Preroll Skywalker OG 1G Sativa,Raw Pre-Rolls,010625SWOG-HB7,2025-04-05 to 2025-04-11,08/06/2025,https://masscannabiscontrol.com/news/public-he...,bar-0 CSV dt-button buttons-csv buttons-html5,8a53b612d4922796.csv,https://masscannabiscontrol.com/news/public-he...,unreviewed,unreviewed
4,Massachusetts,csv_row,CSV,1A40A01000007D1000039528,2025-03-31,2025-02-03,M00000599297: SHORTIES 5PK S/I/H VARIETY PACK ...,Raw Pre-Rolls,"010625SWOG-HB7, 061024PBB-HB8, 070124KM-HB9",2025-04-09 to 2025-04-30,08/06/2025,https://masscannabiscontrol.com/news/public-he...,bar-0 CSV dt-button buttons-csv buttons-html5,8a53b612d4922796.csv,https://masscannabiscontrol.com/news/public-he...,unreviewed,unreviewed


### Check your first result before adding a second state
1. Compare one package or batch ID and its date against the source. Open CSVs with `pd.read_csv(path, dtype=str, keep_default_na=False)` to keep leading zeros.
2. Compare the row count with the website's **total** after clearing filters. An export or a rendered table can contain just the current page. Site-wide totals and equivalence between different file formats cannot be inferred reliably by the generic collector.
3. Inspect the source label, raw file, page number, and record type. The same notice can appear in several formats. PDF pages and their extracted table rows intentionally overlap. HTML text is evidence for review. It may contain navigation or several notices in one block.
4. Review the notice for product origin and action type. If the agency does not identify the origin, keep it unknown. Use distinct agency notice IDs or reviewed event identifiers to count recalls.
5. Add another state or additional notice URLs in section 2 and rerun. If a state publishes a notice archive, add its additional index/detail URLs explicitly. If a page only links to notices, check the `detail_url` column and add the relevant URLs to collect their contents.